# 원본 실험 기록
원본 파일: Music_gen/Creating_data_set_from_midifiles.ipynb

실행 출력과 메타데이터를 제거하고 Drive 경로를 /content/project로 치환했습니다. 셀 순서와 원본 코드를 보존하므로 위에서 아래로 실행이 보장되지 않습니다. 정리된 실행 흐름은 상위 폴더의 01–05 노트북을 참고하세요.

In [ ]:
pip install mido

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
from mido import MidiFile

In [ ]:
mid = MidiFile('/content/project/Data/Boram/Boram_Foolish Games:You Were Meant For Me_Jewel.mid')
# mid = MidiFile('/content/project/Data/Boram/asdf.mid', ticks_per_beat=480)

In [ ]:
print(mid.tracks[0][8])

In [ ]:
for i, track in enumerate(mid.tracks):
    print('Track {}: {}'.format(i, track.name))
    for msg in track:
        print(msg)

In [ ]:
mid.ticks_per_beat

In [ ]:
def read_midi_file(midi_file_path, ticks_per_beat=480):
    # print('midi_file_path', midi_file_path)
    mid = MidiFile(midi_file_path, ticks_per_beat=ticks_per_beat)
    note_temp = []
    note_import = []
    tick = 0
    for i, track in enumerate(mid.tracks):
        for msg in track:
            tick += msg.time
            if msg.type == 'note_on' or msg.type == 'note_off':
                if msg.type == 'note_on'and msg.velocity !=0:
                    pitch = msg.note
                    start = tick*4 / float(mid.ticks_per_beat)
                    start = float("{:.4f}".format(start))  # 시간 퀀타이즈
                    velocity = msg.velocity
                    note_info = {
                        'pitch': pitch,
                        'start_time': start,
                        'length': None,
                        'velocity': velocity,
#                         'function': None,
                    }
                    note_temp.append(note_info)
                elif msg.type == 'note_off'or msg.velocity == 0:
                    for j in range(len(note_temp)):
                        if note_temp[j]['pitch'] == msg.note:
                            end = tick*4 / float(mid.ticks_per_beat)
                            note_temp[j]['length'] = end - note_temp[j]['start_time']
                            note_import.append(note_temp[j])
                            del note_temp[j]
                            break
            else:
                pass
                # print(msg)

    note_import = sorted(note_import, key=lambda k: k['start_time'])
    return note_import

In [ ]:
# read_midi_file('/content/project/Data/Boram/asdfg.mid')
read_midi_file('/content/project/Data/Boram/Boram_Foolish Games:You Were Meant For Me_Jewel.mid')

In [ ]:
li = read_midi_file("/content/project/Data/Haewan/02.junghaewan_I Don't Want To Wait_Paula Cole.mid")
li

In [ ]:
li[0]['pitch']

In [ ]:
import numpy as np
import glob
import random
import matplotlib.pyplot as plt

In [ ]:
#제거할 피치 리스트
remove_these_rows = []
for i in range(36):
  remove_these_rows.append(i)
for i in range(60,72):
  remove_these_rows.append(i)
for i in range(112, 128):
  remove_these_rows.append(i)

remove_these_rows

In [ ]:
read_midi_file(pathes_list[2])

In [ ]:
song = read_midi_file(pathes_list[1])
temp_data = np.zeros((128,128))
if song[-1]['start_time'] >= 128: #9마디 짜리이면
    for note in song:
      if not note['start_time'] < 16.0: #못 갖춘 마디 버림
        x = int(note['pitch'])
        y = int(round(note['start_time']))
        length = int(round(note['length']))
        for i in range(length):
          if y+i-16 > 127:
              False
          else:
            temp_data[x][y+i-16] = 1
else: #8마디 짜리이면
    for note in song:
      x = int(note['pitch'])
      y = int(round(note['start_time']))
      length = int(round(note['length']))
      for i in range(length):
        if y+i > 127:
            pass
        else:
          temp_data[x][y+i] = 1


temp_data = np.delete(temp_data, remove_these_rows, axis=0)
temp_data1 = np.delete(temp_data, np.s_[64::1], axis=1)
temp_data2 = np.delete(temp_data, np.s_[:64:1], axis=1)
print(temp_data1.shape, temp_data2.shape)

plt.imshow(temp_data, cmap='gray', origin='lower')

In [ ]:
np.set_printoptions(threshold=np.inf, linewidth=np.inf)
temp_data

In [ ]:
pathes = '/content/project/Data/*/*.mid'

pathes_list = glob.glob(pathes)

# random.shuffle(pathes_list)

In [ ]:
def make_data_set_128(pathes_list):
  data_set = []

  for path in pathes_list:
    song = read_midi_file(path)
    temp_data = np.zeros((128,128))
    if song[-1]['start_time'] >= 128: #9마디 짜리이면
      for note in song:
        if not note['start_time'] < 16.0: #못 갖춘 마디 버림
          x = int(note['pitch'])
          y = int(round(note['start_time']))
          length = int(round(note['length']))
          for i in range(length):
            if y+i-16 > 127:
              pass
            else:
              temp_data[x][y+i-16] = 1
    else: #8마디 짜리이면
      for note in song:
        x = int(note['pitch'])
        y = int(round(note['start_time']))
        length = int(round(note['length']))
        for i in range(length):
          if y+i > 127:
            pass
          else:
            temp_data[x][y+i] = 1
    
    data_set.append(temp_data)

  data_set = np.array(data_set)
  
  return data_set

In [ ]:
def make_data_set_64(pathes_list):
  data_set = []

  for path in pathes_list:
    song = read_midi_file(path)
    temp_data = np.zeros((128,128))
    if song[-1]['start_time'] >= 128: #9마디 짜리이면
      for note in song:
        if not note['start_time'] < 16.0: #못 갖춘 마디 버림
          x = int(note['pitch'])
          y = int(round(note['start_time']))
          length = int(round(note['length']))
          for i in range(length):
            if y+i-16 > 127:
              pass
            else:
              temp_data[x][y+i-16] = 1
    else: #8마디 짜리이면
      for note in song:
        x = int(note['pitch'])
        y = int(round(note['start_time']))
        length = int(round(note['length']))
        for i in range(length):
          if y+i > 127:
            pass
          else:
            temp_data[x][y+i] = 1
    temp_data = np.delete(temp_data, remove_these_rows, axis=0)
    temp_data1 = np.delete(temp_data, np.s_[64::1], axis=1)
    temp_data2 = np.delete(temp_data, np.s_[:64:1], axis=1)
    data_set.append(temp_data1)
    data_set.append(temp_data2)

  data_set = np.array(data_set)
  
  return data_set

In [ ]:
len(pathes_list)
data_set.shape

In [ ]:
pathes_list[0].split('/')[-2]

In [ ]:
data_set = make_data_set_128(pathes_list)

In [ ]:
from IPython import display
display.clear_output(wait=True)

fig = plt.figure(figsize=(10,10))

for i in range(data_set.shape[0]):
    
    plt.xlim([0, 127])      # X축의 범위: [xmin, xmax]
    plt.ylim([0, 127])     # Y축의 범위: [ymin, ymax]
    plt.xticks([0, 63, 127])
    plt.yticks([0, 35, 60, 71, 112, 127])
    plt.grid(True)
    display.clear_output(wait=True)
    
    plt.imshow(data_set[i, :, :] * 127.5 + 127.5, cmap='gray', origin='lower')
    
    plt.savefig("/content/project/check/{dir}/{name}.png".format(
                                                                      dir = pathes_list[i].split('/')[-2], 
                                                                      name = pathes_list[i].split('/')[-1])
    )

plt.show()

# plt.imshow(data_set[0, :, :] * 127.5 + 127.5, cmap='gray')

In [ ]:
data_set = make_data_set_64(pathes_list)
data_set.shape

In [ ]:
data_set.shape

In [ ]:
plt.imshow(data_set[150], cmap='gray', origin='lower')

In [ ]:
np.set_printoptions(threshold=np.inf, linewidth=np.inf)
data_set[180]

In [ ]:
np.save('/content/project/Data/data_set/data_set_64_sil_jeon', data_set)

In [ ]:
data = np.load('/content/project/Data/data_set/data_set.npy')
plt.imshow(data[180], cmap='gray', origin='lower')